<span style="color:lightgreen">

# L4.1 Transcripción de portugués EuroparlST

Ya que en este notebook no se realizan traducciones, no tiene cabida reporta COMET y BLEU

<span>

# ASR Baseline experiment using Whisper and Europarl-ST (Spanish to English)

In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for ASR on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST) (using the Spanish-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing Word Error Rate, WER)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

import jiwer

model = whisper.load_model("base")

Load Europarl-ST speech dataset

In [9]:
audios = []

with open(CONFIG.europarl_dev_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        audios.append(str(linea).strip())

len(audios)

73

<p style="page-break-after:always;"></p>

Transcribe all the audio data using the Whisper (base) model. The ASR output is stored in hypotheses. At the same time, transcription references are stored into references and translation references into translations.

In [10]:
from maikol_utils.print_utils import print_error
hypotheses = []
references = []
translations = []

for audio in tqdm(audios):
    try:
        with open(os.path.join(CONFIG.europarl_dev_folder, audio, "transcription.tok"), "r", encoding="utf-8") as referencia:
            references.append(referencia.read())
        with open(os.path.join(CONFIG.europarl_dev_folder, audio, "translation_en"), "r", encoding="utf-8") as translation:
            translations.append(translation.read())

        hypotheses.append((model.transcribe(os.path.join(CONFIG.europarl_dev_folder, audio, "audio_clip_diarization.m4a"), language=CONFIG.src_name))['text'])
    except Exception as e:
        print_error(f"Error processing {audio}: {e}")
        hypotheses.append("")
        references.append("")
        translations.append("")

  0%|          | 0/73 [00:00<?, ?it/s]

<p style="page-break-after:always;"></p>

Transcription hypotheses, references and translations are stored into a Pandas dataframe. Show the two first hypotheses.

In [11]:
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references, translation=translations))
pd.set_option('display.max_colwidth', None)
data.head(2)

,hypothesis,reference,translation
0,"Vamos ver. Você sabe que dentro das facultades da presidência e meus colegas vice-pecientes, frequentemente do Affin, está ele realizar comentários e não cu, sobre incidencias de a câmera. E, em estes comentários, você precisa ver o contenido e a intenção. A minha intenção, eu posso segurar que o colegas, que era absolutamente positiva. E, qualquer caso, se você ou outro colegas, já visto este comentário inocente e venê-lo, alguma questão que podem molestar, de você, por retirado por minha parte.","Usted sabe que dentro de las facultades de la Presidencia — y mis colegas Vicepresidentes frecuentemente lo hacen — está el realizar comentarios inocuos sobre incidencias en la Cámara , y en estos comentarios hay que ver el contenido y la intención .\nLe puedo asegurar , querido colega , que mi intención era absolutamente positiva .\nEn cualquier caso , si usted o algún otro colega ha visto en este comentario inocente y benévolo alguna cuestión que les pueda molestar , delo usted por retirado por mi parte .\n","You must be aware that one of the President’s powers – and my Vice-President colleagues frequently do this – is to make innocuous comments about incidents in the House. These comments must be taken in the context of their content and intention.\nI can assure you, dear Member, that my intention was absolutely positive.\nHowever, if you or any other Member has found something disturbing in this innocent and benevolent comment, you must regard it as having been withdrawn."
1,"Gracias, señor presidente. Hace exactamente um año, o pueblo venezolano rechazou em referendum que Hugo Chávez alargará su mandato presidencial, fijado na Constituição venezolana. Pois bem, Hugo Chávez, haciendo caso o misso, a lo que o pueblo soberano decidiu democráticamente, ha anunciado que vai cambiar as leyes com o fim de para petarse em poder. Conegio, Hugo Chávez demuestra uma vez más que não é um presidente democrático, sin um caudillo, um distador golpista, cujo objetivo é convertir a toda venezuela em su rancho particular. E, de esta maneira, seguir amenazando, insultando e agrediendo a oposição e a dissidência. Assim como também, sua intención é seguir dinamitando a liberta desfresión, cerrando medios de comunicação como hizo com o radio Caracas Televisión. Este Parlamento europeu deve condenar e rechazar energicamente los trujos e artimanhas que Hugo Chávez pretende poner em marcha para não abandonar a presidencia del país. E animamos a la sociedad venezolana a defender os valores democráticos e de libertas que estão em as antípadas de o que dizem o Chávez.","Señor Presidente , hace exactamente un año , el pueblo venezolano rechazó en referéndum que Hugo Chávez alargara su mandato presidencial fijado en la Constitución venezolana .\nPues bien , Hugo Chávez , haciendo caso omiso a lo que el pueblo soberano decidió democráticamente , ha anunciado que va a cambiar las leyes con el fin de perpetuarse en el poder .\nCon ello , Hugo Chávez demuestra una vez más que no es un presidente democrático sino un caudillo , un dictador golpista cuyo objetivo es convertir a toda Venezuela en su rancho particular y , de esta manera , seguir amenazando , insultando y agrediendo a la oposición y a la disidencia , así como también es su intención seguir dinamitando la libertad de expresión , cerrando medios de comunicación como hizo con Radio Caracas Televisión .\nEste Parlamento Europeo debe condenar y rechazar enérgicamente los trucos y las artimañas que Hugo Chávez pretende poner en marcha para no abandonar la presidencia del país y animamos a la sociedad venezolana a defender los valores democráticos y de libertad , que están en las antípodas de lo que hace y dice Hugo Chávez .\n","Mr President, exactly one year ago the people of Venezuela voted in a referendum that Hugo Chávez should not extend his term of office as president, which is fixed under the Venezuelan constitution.\nWell, Hugo Chávez has ignored the sovereign pe

Hypotheses and references are normalized using the Whisper Basic text standardisation/normalization module

In [12]:
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["translation_clean"] = [normalizer(text) for text in data["translation"]]
data.head(2)

,hypothesis,reference,translation,hypothesis_clean,reference_clean,translation_clean
0,"Vamos ver. Você sabe que dentro das facultades da presidência e meus colegas vice-pecientes, frequentemente do Affin, está ele realizar comentários e não cu, sobre incidencias de a câmera. E, em estes comentários, você precisa ver o contenido e a intenção. A minha intenção, eu posso segurar que o colegas, que era absolutamente positiva. E, qualquer caso, se você ou outro colegas, já visto este comentário inocente e venê-lo, alguma questão que podem molestar, de você, por retirado por minha parte.","Usted sabe que dentro de las facultades de la Presidencia — y mis colegas Vicepresidentes frecuentemente lo hacen — está el realizar comentarios inocuos sobre incidencias en la Cámara , y en estos comentarios hay que ver el contenido y la intención .\nLe puedo asegurar , querido colega , que mi intención era absolutamente positiva .\nEn cualquier caso , si usted o algún otro colega ha visto en este comentario inocente y benévolo alguna cuestión que les pueda molestar , delo usted por retirado por mi parte .\n","You must be aware that one of the President’s powers – and my Vice-President colleagues frequently do this – is to make innocuous comments about incidents in the House. These comments must be taken in the context of their content and intention.\nI can assure you, dear Member, that my intention was absolutely positive.\nHowever, if you or any other Member has found something disturbing in this innocent and benevolent comment, you must regard it as having been withdrawn.",vamos ver você sabe que dentro das facultades da presidência e meus colegas vice pecientes frequentemente do affin está ele realizar comentários e não cu sobre incidencias de a câmera e em estes comentários você precisa ver o contenido e a intenção a minha intenção eu posso segurar que o colegas que era absolutamente positiva e qualquer caso se você ou outro colegas já visto este comentário inocente e venê lo alguma questão que podem molestar de você por retirado por minha parte,usted sabe que dentro de las facultades de la presidencia y mis colegas vicepresidentes frecuentemente lo hacen está el realizar comentarios inocuos sobre incidencias en la cámara y en estos comentarios hay que ver el contenido y la intención le puedo asegurar querido colega que mi intención era absolutamente positiva en cualquier caso si usted o algún otro colega ha visto en este comentario inocente y benévolo alguna cuestión que les pueda molestar delo usted por retirado por mi parte,you must be aware that one of the president s powers and my vice president colleagues frequently do this is to make innocuous comments about incidents in the house these comments must be taken in the context of their content and intention i can assure you dear member that my intention was absolutely positive however if you or any other member has found something disturbing in this innocent and benevolent comment you must regard it as having been withdrawn
1,"Gracias, señor presidente. Hace exactamente um año, o pueblo venezolano rechazou em referendum que Hugo Chávez alargará su mandato presidencial, fijado na Constituição venezolana. Pois bem, Hugo Chávez, haciendo caso o misso, a lo que o pueblo soberano decidiu democráticamente, ha anunciado que vai cambiar as leyes com o fim de para petarse em poder. Conegio, Hugo Chávez demuestra uma vez más que não é um presidente democrático, sin um caudillo, um distador golpista, cujo objetivo é convertir a toda venezuela em su rancho particular. E, de esta maneira, seguir amenazando, insultando e agrediendo a oposição e a dissidência. Assim como também, sua intención é seguir dinamitando a liberta desfresión, cerrando medios de comunicação como hizo com o radio Caracas Televisión. Este Parlamento europeu deve condenar e rechazar energicamente los trujos e artimanhas que Hugo Chávez pretende poner em marcha para não abandonar a presidencia del país. E animamos a la sociedad ve

Finally, we compute the transcription WER using [JIWER](https://openai.com/index/whisper/) which is a simple and fast python package to evaluate ASR performance.

In [13]:

wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))

print(f"WER: {wer * 100:.2f} %")

WER: 57.13 %


All the data is stored into a file using 'csv' format

In [14]:
data.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'L4.1_ASR_Whisper_Baseline_dev_Europarl-ST.csv'), encoding='utf-8')